# Step 17 — Deep review of tumor/unresolved Leiden resolution 0.2

This notebook **does not rerun PCA, Harmony, neighbors, UMAP, or Leiden**. It
loads the completed Step 16 cancer-specific objects and performs an
interpretation-focused review of:

```python
obs["tumor_leiden_res_0_2"]
```

for:

```text
melanoma
NSCLC
colon_cancer
```

## Why resolution 0.2?

Resolution 0.2 is treated as the primary interpretive resolution because it
provides a manageable number of broad tumor/epithelial states. Higher
resolutions remain available in the source objects, but are used here only for
a nested-resolution comparison with resolution 0.6.

## What is repeated from the previous resolution 0.6 review?

For resolution 0.2, the notebook generates:

- balanced Wilcoxon marker ranking;
- top-marker tables;
- raw-count support for ranked markers;
- cancer-specific marker dotplots;
- UMAPs labeled by cluster, sample, biopsy stage, and original broad label;
- spatial cluster maps for every sample;
- cluster size and sample/patient composition summaries.

## Additional tumor-biology review

The notebook also evaluates compact, exploratory programs related to:

- lineage differentiation and lineage fidelity;
- stem/progenitor-like or dedifferentiated states;
- epithelial–mesenchymal transition and matrix invasion;
- hypoxia;
- immediate-early/AP-1 activation;
- heat-shock/proteotoxic stress;
- unfolded-protein/ER stress;
- oxidative/NRF2-associated stress;
- p53/DNA-damage response;
- interferon/inflammatory response;
- proliferation.

### Important interpretation limit

These are **transcriptional state programs**, not validated pathology grading
classifiers. A cluster with low lineage differentiation and high stem/EMT
programs may be compatible with a less differentiated or more plastic state,
but it should not be labeled “poorly differentiated carcinoma” without
histology and pathology review.

The marker panels are used only after unsupervised clustering. They do not
alter the existing Leiden assignments.


## Scientific basis for the exploratory panels

The compact programs are informed by several established observations:

- melanoma cells occupy lineage-differentiation states ranging from
  MITF/melanocytic to AXL-associated dedifferentiated states, with intermediate
  neural-crest-like states;
- LUAD epithelial progression can involve loss of alveolar lineage identity
  and acquisition of stem-like features, while NSCLC also contains distinct
  glandular, squamous/basal, and airway differentiation programs;
- colorectal tumors retain multilineage intestinal differentiation while also
  containing tumor-specific stem/progenitor-like states;
- hypoxia, ER stress, oxidative stress, heat shock, DNA damage, and
  immediate-early/AP-1 activation are separable stress-response programs.

References are listed at the end of the notebook. The exact genes present in
your Visium HD object are audited before any score or dotplot is created.


In [1]:
# ---------------------------------------------------------------------
# Imports and environment
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import json
import math
import re
import shutil
import time
import traceback
import warnings
from collections import OrderedDict
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
plt.ioff()

print("Python:", __import__("sys").executable)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


Python: /home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
anndata: 0.12.11
scanpy: 1.12.1
numpy: 2.4.4
pandas: 2.3.3


In [2]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
PIPELINE_ROOT = (
    PROJECT_ROOT
    / "tmp"
    / "proseg_resolvi_immune_enrichment_v1"
)

INPUT_ROOT = (
    PIPELINE_ROOT
    / "16_tumor_unresolved_leiden_multiresolution_selectionfix_v2"
)
OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "17_tumor_unresolved_resolution0p2_deep_review"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CANCER_TYPES_TO_RUN = [
    "melanoma",
    "NSCLC",
    "colon_cancer",
]

PRIMARY_RESOLUTION = 0.2
COMPARISON_RESOLUTION = 0.6

CLUSTER_KEY = "tumor_leiden_res_0_2"
COMPARISON_CLUSTER_KEY = "tumor_leiden_res_0_6"

ANNOTATION_COLUMN = (
    "prelim_cell_type_primary_tumor_expanded"
)

# Marker ranking.
N_MARKER_GENES = 150
MAX_MARKER_CELLS_PER_CLUSTER = 5_000
TOP_MARKERS_PER_CLUSTER_FOR_DOTPLOT = 8
MARKER_PADJ_MAX = 0.05
MARKER_LOGFC_MIN = 0.25
MARKER_RAW_DETECTION_MIN = 0.05
MARKER_RAW_DETECTION_DELTA_MIN = 0.02

# Program scoring.
MIN_PROGRAM_GENES = 3
MIN_PROGRAM_COVERAGE_FRACTION = 0.25
PROGRAM_SCORE_CLIP_Z = 8.0

# Plotting.
PLOT_DPI = 320
PLOT_MAX_CELLS = 250_000
UMAP_POINT_SIZE = 1.0
SPATIAL_POINT_SIZE = 0.45
SPATIAL_SCORE_POINT_SIZE = 0.45

# Output choices.
WRITE_CELL_SCORE_PARQUET = True
WRITE_ENRICHED_H5AD = False
H5AD_COMPRESSION = "lzf"

OVERWRITE = False
CONTINUE_ON_ERROR = True
RANDOM_STATE = 0

PIPELINE_VERSION = (
    "2026-08-20-tumor-unresolved-resolution0p2-deep-review-v1"
)

print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Cancer types:", CANCER_TYPES_TO_RUN)
print("Primary cluster key:", CLUSTER_KEY)


Input root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/17_tumor_unresolved_resolution0p2_deep_review
Cancer types: ['melanoma', 'NSCLC', 'colon_cancer']
Primary cluster key: tumor_leiden_res_0_2


In [3]:
# ---------------------------------------------------------------------
# Input and output paths
# ---------------------------------------------------------------------
def cancer_slug(cancer_type: str) -> str:
    return (
        str(cancer_type)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )


def input_paths(cancer_type: str) -> dict[str, Path]:
    slug = cancer_slug(cancer_type)
    root = INPUT_ROOT / slug
    prefix = f"{slug}_tumor_unresolved"

    return {
        "root": root,
        "zarr": root / f"{prefix}_multires_leiden.zarr",
        "h5ad": root / f"{prefix}_multires_leiden.h5ad",
        "success": root / f"{prefix}_SUCCESS.json",
    }


def output_paths(cancer_type: str) -> dict[str, Path]:
    slug = cancer_slug(cancer_type)
    root = OUTPUT_ROOT / slug
    figures = root / "figures"
    tables = root / "tables"
    markers = root / "markers"
    spatial = figures / "spatial"

    for path in [root, figures, tables, markers, spatial]:
        path.mkdir(parents=True, exist_ok=True)

    prefix = f"{slug}_resolution0p2"
    return {
        "root": root,
        "figures": figures,
        "tables": tables,
        "markers": markers,
        "spatial": spatial,
        "cluster_summary": (
            tables / f"{prefix}_cluster_summary.csv"
        ),
        "sample_composition": (
            tables / f"{prefix}_cluster_sample_composition.csv"
        ),
        "patient_composition": (
            tables / f"{prefix}_cluster_patient_composition.csv"
        ),
        "stage_composition": (
            tables / f"{prefix}_cluster_stage_composition.csv"
        ),
        "source_label_composition": (
            tables / f"{prefix}_cluster_source_label_composition.csv"
        ),
        "marker_table": (
            markers / f"{prefix}_balanced_markers.csv"
        ),
        "marker_table_supported": (
            markers / f"{prefix}_balanced_markers_raw_supported.csv"
        ),
        "top_marker_dotplot_table": (
            markers / f"{prefix}_top_marker_hybrid_dotplot_table.csv"
        ),
        "program_availability": (
            tables / f"{prefix}_program_gene_availability.csv"
        ),
        "program_cluster_scores": (
            tables / f"{prefix}_program_scores_by_cluster.csv"
        ),
        "program_raw_detection": (
            tables / f"{prefix}_program_raw_detection_by_cluster.csv"
        ),
        "program_cell_scores": (
            tables / f"{prefix}_cell_program_scores.parquet"
        ),
        "annotation_workbench": (
            tables / f"{prefix}_cluster_annotation_workbench.csv"
        ),
        "res02_vs_res06_counts": (
            tables / f"{prefix}_vs_resolution0p6_counts.csv"
        ),
        "res02_vs_res06_fraction": (
            tables / f"{prefix}_vs_resolution0p6_row_fraction.csv"
        ),
        "umap_audit": (
            figures / f"{prefix}_umap_audit.png"
        ),
        "umap_programs_lineage": (
            figures / f"{prefix}_umap_lineage_program_scores.png"
        ),
        "umap_programs_stress": (
            figures / f"{prefix}_umap_stress_program_scores.png"
        ),
        "biology_dotplot_lineage": (
            figures / f"{prefix}_lineage_differentiation_hybrid_dotplot.png"
        ),
        "biology_dotplot_stress": (
            figures / f"{prefix}_invasion_stress_hybrid_dotplot.png"
        ),
        "top_marker_dotplot": (
            figures / f"{prefix}_top_markers_hybrid_dotplot.png"
        ),
        "program_heatmap": (
            figures / f"{prefix}_program_score_heatmap.png"
        ),
        "nested_resolution_heatmap": (
            figures / f"{prefix}_vs_resolution0p6_heatmap.png"
        ),
        "summary": root / f"{prefix}_summary.json",
        "success": root / f"{prefix}_SUCCESS.json",
        "failure": root / f"{prefix}_FAILURE.json",
        "enriched_h5ad": root / f"{prefix}_deep_review.h5ad",
    }


def atomic_write_json(payload: dict, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, default=str)
    )
    temporary.replace(path)


def natural_cluster_order(values) -> list[str]:
    unique = pd.Index(values.astype(str).unique())

    def sort_key(value):
        pieces = re.split(r"(\d+)", str(value))
        return tuple(
            int(piece) if piece.isdigit() else piece
            for piece in pieces
        )

    return sorted(unique.tolist(), key=sort_key)


preflight_rows = []
for cancer_type in CANCER_TYPES_TO_RUN:
    paths = input_paths(cancer_type)
    source = (
        paths["zarr"]
        if paths["zarr"].exists()
        else paths["h5ad"]
    )
    preflight_rows.append(
        {
            "cancer_type": cancer_type,
            "zarr_exists": paths["zarr"].exists(),
            "h5ad_exists": paths["h5ad"].exists(),
            "success_exists": paths["success"].exists(),
            "selected_source": str(source),
            "source_exists": source.exists(),
        }
    )

preflight = pd.DataFrame(preflight_rows)
display(preflight)

if not preflight["source_exists"].all():
    raise FileNotFoundError(
        "One or more completed Step 16 cancer objects are missing."
    )


,cancer_type,zarr_exists,h5ad_exists,success_exists,selected_source,source_exists
0,melanoma,True,False,True,/host_root/nethome/reny28/Projects/Visium_proj...,True
1,NSCLC,True,False,True,/host_root/nethome/reny28/Projects/Visium_proj...,True
2,colon_cancer,True,False,True,/host_root/nethome/reny28/Projects/Visium_proj...,True


## Exploratory tumor-biology programs

The lists below are deliberately compact so they remain interpretable in a
spatial dataset. Programs are evaluated only when at least three genes and at
least 25% of the requested genes are present.

The notebook writes a complete availability table containing:

```text
requested genes
present genes
missing genes
coverage fraction
whether the program was scored
```

### Program naming

- “Differentiated” means stronger lineage-associated transcription, not a
  pathology grade.
- “Stem/progenitor” or “dedifferentiated” means a transcriptional state
  associated with lineage plasticity; it does not prove cancer stem-cell
  function.
- “Invasion/EMT” is an expression program, not direct evidence of metastasis.


In [4]:
# ---------------------------------------------------------------------
# Curated exploratory gene programs
# ---------------------------------------------------------------------
PAN_CANCER_PROGRAMS = OrderedDict(
    {
        "Pan | Proliferation": [
            "MKI67", "TOP2A", "UBE2C", "BIRC5", "CCNB1",
            "CDC20", "CENPF", "PCNA", "MCM2", "MCM5",
        ],
        "Pan | EMT / mesenchymal": [
            "VIM", "FN1", "ZEB1", "ZEB2", "SNAI1",
            "SNAI2", "TWIST1", "ITGA5", "TGFBI", "SERPINE1",
        ],
        "Pan | Matrix invasion / motility": [
            "MMP2", "MMP7", "MMP9", "MMP14", "PLAU",
            "PLAUR", "CTSB", "CTSD", "CXCR4", "LAMC2",
        ],
        "Pan | Hypoxia": [
            "CA9", "VEGFA", "SLC2A1", "LDHA", "PDK1",
            "BNIP3", "NDRG1", "EGLN3", "ADM",
        ],
        "Pan | Immediate-early / AP-1": [
            "FOS", "FOSB", "JUN", "JUNB", "JUND",
            "ATF3", "EGR1", "DUSP1", "DUSP5",
        ],
        "Pan | Heat shock / proteotoxic": [
            "HSPA1A", "HSPA1B", "HSPH1", "DNAJB1",
            "HSP90AA1", "HSPB1", "HSPD1", "HSPE1",
        ],
        "Pan | UPR / ER stress": [
            "HSPA5", "XBP1", "DDIT3", "ATF4", "HERPUD1",
            "DNAJB9", "PDIA4", "PPP1R15A", "TRIB3",
        ],
        "Pan | Oxidative / NRF2-associated": [
            "HMOX1", "NQO1", "GCLC", "GCLM", "TXNRD1",
            "SLC7A11", "SRXN1",
        ],
        "Pan | p53 / DNA damage": [
            "CDKN1A", "MDM2", "GADD45A", "DDB2", "SESN1",
            "BBC3", "PMAIP1", "BAX", "TP53INP1",
        ],
        "Pan | Interferon / inflammatory": [
            "IFIT1", "IFIT2", "IFIT3", "ISG15", "MX1",
            "OAS1", "OAS2", "STAT1", "IRF1", "CXCL10",
        ],
    }
)

CANCER_SPECIFIC_PROGRAMS = {
    "melanoma": OrderedDict(
        {
            "Melanoma | Melanocytic differentiation": [
                "MITF", "MLANA", "PMEL", "TYR", "DCT",
                "TYRP1", "SLC45A2", "OCA2", "GPNMB", "PAX3",
            ],
            "Melanoma | Neural-crest / transitory": [
                "SOX10", "NGFR", "ERBB3", "GFRA2", "FABP7",
                "PLP1", "S100B", "SOX9",
            ],
            "Melanoma | AXL / dedifferentiated": [
                "AXL", "WNT5A", "ZEB1", "SMAD3", "FOSL1",
                "JUN", "VIM", "FN1", "ITGA5", "TGFBI",
            ],
            "Melanoma | Stratified / keratinocyte-like": [
                "DSP", "DMKN", "DSG1", "DSC3", "KRT5",
                "KRT14", "KRT17", "KRT6A", "KRT16", "SFN",
            ],
        }
    ),
    "NSCLC": OrderedDict(
        {
            "NSCLC | AT2 / alveolar lineage": [
                "NKX2-1", "SFTPA1", "SFTPA2", "SFTPB", "SFTPC",
                "NAPSA", "ABCA3", "SLC34A2", "CLDN18", "LPCAT1",
            ],
            "NSCLC | AT1-like maturation": [
                "AGER", "CAV1", "CAV2", "EMP2", "PDPN",
                "HOPX", "AQP5",
            ],
            "NSCLC | Glandular / adenocarcinoma": [
                "KRT7", "EPCAM", "KRT8", "KRT18", "KRT19",
                "MUC1", "CEACAM5", "CEACAM6", "TFF3", "MSLN",
            ],
            "NSCLC | Squamous / basal": [
                "TP63", "KRT5", "KRT14", "KRT17", "KRT6A",
                "KRT6B", "DSG3", "DSC3", "SFN", "SOX2",
            ],
            "NSCLC | Club / secretory airway": [
                "SCGB1A1", "SCGB3A1", "KRT4", "KRT13",
                "CYP2F1", "KLF5",
            ],
            "NSCLC | Ciliated airway": [
                "FOXJ1", "PIFO", "CAPS", "TPPP3",
                "CCDC78", "CFAP54",
            ],
            "NSCLC | Progenitor / lineage-loss-associated": [
                "MDK", "TIMP1", "IFI27", "S100A4",
                "SOX9", "HMGA2", "PROM1", "ALDH1A1",
            ],
        }
    ),
    "colon_cancer": OrderedDict(
        {
            "CRC | Enterocyte / absorptive differentiation": [
                "CDX2", "KRT20", "VIL1", "ALPI", "CA1", "CA2",
                "FABP1", "FABP2", "SI", "SLC26A3", "GUCA2A",
                "HNF4A", "HNF4G",
            ],
            "CRC | Goblet / secretory differentiation": [
                "MUC2", "FCGBP", "TFF3", "SPINK4", "REG4",
                "AGR2", "CLCA1", "ZG16", "SPDEF", "ATOH1",
            ],
            "CRC | Stem / progenitor-like": [
                "LGR5", "ASCL2", "OLFM4", "SOX9", "PROM1",
                "LRIG1", "SMOC2", "EPHB2", "MYC", "AXIN2",
            ],
            "CRC | Regenerative / lineage-plasticity-associated": [
                "ANXA1", "ANXA3", "CLDN4", "KRT17",
                "SOX9", "YAP1", "SPP1", "PLAUR",
            ],
            "CRC | Paneth-like / antimicrobial": [
                "DEFA5", "DEFA6", "LYZ", "REG1A",
                "REG3A", "REG4",
            ],
        }
    ),
}

COMPETING_LINEAGE_AUDIT = OrderedDict(
    {
        "Audit | Immune": [
            "PTPRC", "LST1", "TYROBP", "CD3D", "NKG7",
        ],
        "Audit | Fibroblast / stromal": [
            "COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "COL6A1",
        ],
        "Audit | Endothelial": [
            "PECAM1", "VWF", "EMCN", "KDR", "ENG",
        ],
    }
)


def program_slug(name: str) -> str:
    return (
        re.sub(r"[^A-Za-z0-9]+", "_", name)
        .strip("_")
        .lower()
    )


print(
    "Programs per cancer:",
    {
        cancer: len(programs)
        for cancer, programs in CANCER_SPECIFIC_PROGRAMS.items()
    },
)


Programs per cancer: {'melanoma': 4, 'NSCLC': 7, 'colon_cancer': 5}


In [5]:
# ---------------------------------------------------------------------
# Load completed Step 16 cancer object
# ---------------------------------------------------------------------
def load_cancer_object(cancer_type: str) -> ad.AnnData:
    paths = input_paths(cancer_type)

    if paths["zarr"].exists():
        print("Loading Zarr:", paths["zarr"])
        if hasattr(ad.experimental, "read_lazy"):
            lazy = ad.experimental.read_lazy(
                str(paths["zarr"])
            )
            candidate = lazy.to_memory()
        else:
            candidate = ad.read_zarr(paths["zarr"])
    elif paths["h5ad"].exists():
        print("Loading H5AD:", paths["h5ad"])
        candidate = ad.read_h5ad(paths["h5ad"])
    else:
        raise FileNotFoundError(
            f"No completed Step 16 object for {cancer_type}."
        )

    candidate.obs_names = candidate.obs_names.astype(str)
    candidate.var_names = candidate.var_names.astype(str)
    candidate.var_names_make_unique()

    for required in [
        CLUSTER_KEY,
        "sample",
        "patient",
        "biopsy_stage",
        ANNOTATION_COLUMN,
    ]:
        if required not in candidate.obs:
            raise KeyError(
                f"{cancer_type}: missing obs column {required!r}."
            )

    if "X_umap" not in candidate.obsm:
        raise KeyError(
            f"{cancer_type}: completed object lacks obsm['X_umap']."
        )

    if "counts" not in candidate.layers:
        raise KeyError(
            f"{cancer_type}: completed object lacks layers['counts']."
        )

    raw = candidate.layers["counts"]
    if not sp.issparse(raw):
        raw = sp.csr_matrix(raw)
    else:
        raw = sp.csr_matrix(raw)
    raw.eliminate_zeros()
    raw.sort_indices()
    candidate.layers["counts"] = raw

    corrected = candidate.X
    if hasattr(corrected, "compute"):
        corrected = corrected.compute()
    corrected = np.asarray(
        corrected,
        dtype=np.float32,
    )
    if not np.isfinite(corrected).all():
        raise ValueError(
            f"{cancer_type}: corrected expression has NaN/inf."
        )
    candidate.X = corrected

    candidate.obs[CLUSTER_KEY] = pd.Categorical(
        candidate.obs[CLUSTER_KEY].astype(str),
        categories=natural_cluster_order(
            candidate.obs[CLUSTER_KEY]
        ),
        ordered=True,
    )

    if COMPARISON_CLUSTER_KEY in candidate.obs:
        candidate.obs[
            COMPARISON_CLUSTER_KEY
        ] = pd.Categorical(
            candidate.obs[
                COMPARISON_CLUSTER_KEY
            ].astype(str),
            categories=natural_cluster_order(
                candidate.obs[
                    COMPARISON_CLUSTER_KEY
                ]
            ),
            ordered=True,
        )

    print(candidate)
    print(
        "Resolution 0.2 cluster sizes:",
        candidate.obs[CLUSTER_KEY]
        .value_counts(sort=False)
        .to_dict(),
    )
    return candidate


In [6]:
# ---------------------------------------------------------------------
# Program availability, deterministic scores, and raw detection
# ---------------------------------------------------------------------
def combined_programs(cancer_type: str) -> OrderedDict:
    programs = OrderedDict()
    programs.update(
        CANCER_SPECIFIC_PROGRAMS[cancer_type]
    )
    programs.update(PAN_CANCER_PROGRAMS)
    programs.update(COMPETING_LINEAGE_AUDIT)
    return programs


def audit_program_availability(
    candidate: ad.AnnData,
    cancer_type: str,
) -> tuple[OrderedDict, pd.DataFrame]:
    var_names = set(candidate.var_names.astype(str))
    available = OrderedDict()
    rows = []

    for program, genes in combined_programs(
        cancer_type
    ).items():
        present = [
            gene for gene in genes
            if gene in var_names
        ]
        missing = [
            gene for gene in genes
            if gene not in var_names
        ]
        coverage = (
            len(present) / len(genes)
            if genes else 0.0
        )
        score_program = (
            len(present) >= MIN_PROGRAM_GENES
            and coverage >= MIN_PROGRAM_COVERAGE_FRACTION
        )

        rows.append(
            {
                "cancer_type": cancer_type,
                "program": program,
                "n_requested": len(genes),
                "n_present": len(present),
                "n_missing": len(missing),
                "coverage_fraction": coverage,
                "score_program": score_program,
                "present_genes": ";".join(present),
                "missing_genes": ";".join(missing),
            }
        )

        if score_program:
            available[program] = present

    return available, pd.DataFrame(rows)


def expression_block(
    candidate: ad.AnnData,
    genes: list[str],
) -> np.ndarray:
    values = candidate[:, genes].X
    if sp.issparse(values):
        values = values.toarray()
    if hasattr(values, "compute"):
        values = values.compute()
    return np.asarray(values, dtype=np.float32)


def score_programs(
    candidate: ad.AnnData,
    programs: OrderedDict,
) -> list[str]:
    score_columns = []

    for program, genes in programs.items():
        values = expression_block(
            candidate,
            genes,
        )

        means = values.mean(axis=0)
        scales = values.std(axis=0)
        scales[
            ~np.isfinite(scales)
            | (scales < 1e-6)
        ] = 1.0

        values -= means
        values /= scales
        np.clip(
            values,
            -PROGRAM_SCORE_CLIP_Z,
            PROGRAM_SCORE_CLIP_Z,
            out=values,
        )

        score = values.mean(axis=1)
        column = (
            "program_score__"
            + program_slug(program)
        )
        candidate.obs[column] = score.astype(
            np.float32
        )
        score_columns.append(column)

    return score_columns


def program_cluster_tables(
    candidate: ad.AnnData,
    programs: OrderedDict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    score_rows = []
    raw_rows = []

    cluster_values = (
        candidate.obs[CLUSTER_KEY]
        .astype(str)
        .to_numpy()
    )
    cluster_order = natural_cluster_order(
        candidate.obs[CLUSTER_KEY]
    )

    for program, genes in programs.items():
        score_column = (
            "program_score__"
            + program_slug(program)
        )

        raw = candidate[:, genes].layers[
            "counts"
        ]
        raw = (
            raw.tocsr()
            if sp.issparse(raw)
            else sp.csr_matrix(raw)
        )

        for cluster in cluster_order:
            mask = cluster_values == cluster
            n_cells = int(mask.sum())

            cluster_scores = candidate.obs.loc[
                mask,
                score_column,
            ].to_numpy(dtype=float)

            raw_subset = raw[mask]
            any_gene = (
                np.asarray(
                    raw_subset.sum(axis=1)
                ).reshape(-1)
                > 0
            )
            fraction_genes_detected = np.asarray(
                (raw_subset > 0).mean(axis=1)
            ).reshape(-1)

            score_rows.append(
                {
                    "cluster": cluster,
                    "program": program,
                    "score_column": score_column,
                    "n_cells": n_cells,
                    "mean_program_score": float(
                        np.mean(cluster_scores)
                    ),
                    "median_program_score": float(
                        np.median(cluster_scores)
                    ),
                }
            )

            raw_rows.append(
                {
                    "cluster": cluster,
                    "program": program,
                    "n_cells": n_cells,
                    "n_program_genes": len(genes),
                    "fraction_cells_any_raw_gene": float(
                        any_gene.mean()
                    ),
                    "mean_fraction_program_genes_detected": float(
                        fraction_genes_detected.mean()
                    ),
                }
            )

    return (
        pd.DataFrame(score_rows),
        pd.DataFrame(raw_rows),
    )


In [7]:
# ---------------------------------------------------------------------
# Hybrid dotplot: raw detection for size, corrected expression for color
# ---------------------------------------------------------------------
def hybrid_dotplot_table(
    candidate: ad.AnnData,
    program_groups: OrderedDict,
) -> pd.DataFrame:
    cluster_order = natural_cluster_order(
        candidate.obs[CLUSTER_KEY]
    )
    cluster_values = (
        candidate.obs[CLUSTER_KEY]
        .astype(str)
        .to_numpy()
    )

    rows = []

    for program, genes in program_groups.items():
        present = [
            gene
            for gene in genes
            if gene in candidate.var_names
        ]
        if not present:
            continue

        corrected = expression_block(
            candidate,
            present,
        )
        raw = candidate[
            :,
            present,
        ].layers["counts"]
        raw = (
            raw.tocsr()
            if sp.issparse(raw)
            else sp.csr_matrix(raw)
        )

        for cluster in cluster_order:
            mask = cluster_values == cluster

            corrected_mean = corrected[
                mask
            ].mean(axis=0)
            raw_detection = np.asarray(
                (raw[mask] > 0).mean(axis=0)
            ).reshape(-1)

            for gene, mean_value, detection in zip(
                present,
                corrected_mean,
                raw_detection,
            ):
                rows.append(
                    {
                        "cluster": cluster,
                        "program": program,
                        "gene": gene,
                        "corrected_mean": float(
                            mean_value
                        ),
                        "raw_detection_fraction": float(
                            detection
                        ),
                        "n_cells": int(mask.sum()),
                    }
                )

    table = pd.DataFrame(rows)
    if table.empty:
        return table

    table["corrected_mean_z_within_gene"] = (
        table.groupby("gene")[
            "corrected_mean"
        ]
        .transform(
            lambda values: (
                values - values.mean()
            )
            / (
                values.std(ddof=0)
                if values.std(ddof=0) > 1e-8
                else 1.0
            )
        )
        .clip(-3, 3)
    )
    return table


def save_hybrid_dotplot(
    table: pd.DataFrame,
    program_groups: OrderedDict,
    path: Path,
    title: str,
):
    if table.empty:
        print("Hybrid dotplot skipped; no genes were available.")
        return

    genes = []
    gene_program = {}
    for program, requested in program_groups.items():
        present = [
            gene
            for gene in requested
            if gene in set(table["gene"])
        ]
        for gene in present:
            if gene not in gene_program:
                genes.append(gene)
                gene_program[gene] = program

    clusters = natural_cluster_order(
        table["cluster"]
    )
    gene_position = {
        gene: index
        for index, gene in enumerate(genes)
    }
    cluster_position = {
        cluster: index
        for index, cluster in enumerate(clusters)
    }

    plot_table = table[
        table["gene"].isin(genes)
        & table["cluster"].isin(clusters)
    ].copy()
    plot_table["x"] = plot_table[
        "gene"
    ].map(gene_position)
    plot_table["y"] = plot_table[
        "cluster"
    ].map(cluster_position)

    figure_width = max(
        12,
        0.32 * len(genes),
    )
    figure_height = max(
        4.5,
        0.55 * len(clusters) + 2.5,
    )

    fig, ax = plt.subplots(
        figsize=(figure_width, figure_height)
    )

    sizes = (
        12
        + 230
        * plot_table[
            "raw_detection_fraction"
        ].to_numpy()
    )
    scatter = ax.scatter(
        plot_table["x"],
        plot_table["y"],
        s=sizes,
        c=plot_table[
            "corrected_mean_z_within_gene"
        ],
        cmap="RdBu_r",
        vmin=-2.5,
        vmax=2.5,
        linewidths=0.15,
        edgecolors="black",
    )

    ax.set_xticks(
        range(len(genes)),
        labels=genes,
        rotation=90,
        fontsize=8,
    )
    ax.set_yticks(
        range(len(clusters)),
        labels=clusters,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Leiden cluster (resolution 0.2)")
    ax.set_title(title)

    # Program separators and labels.
    program_ranges = []
    start = 0
    for program, requested in program_groups.items():
        present = [
            gene
            for gene in requested
            if gene in gene_position
        ]
        if not present:
            continue
        positions = [
            gene_position[gene]
            for gene in present
        ]
        left = min(positions)
        right = max(positions)
        program_ranges.append(
            (program, left, right)
        )

    for _, _, right in program_ranges[:-1]:
        ax.axvline(
            right + 0.5,
            linewidth=0.7,
            color="grey",
            alpha=0.6,
        )

    for index, (program, left, right) in enumerate(
        program_ranges
    ):
        ax.text(
            (left + right) / 2,
            -1.15 - 0.22 * (index % 2),
            program,
            ha="center",
            va="top",
            fontsize=7,
            rotation=0,
            transform=ax.transData,
        )

    colorbar = fig.colorbar(
        scatter,
        ax=ax,
        fraction=0.025,
        pad=0.01,
    )
    colorbar.set_label(
        "Corrected mean expression\nz-scored within gene"
    )

    size_handles = []
    for fraction in [0.1, 0.5, 0.9]:
        size_handles.append(
            ax.scatter(
                [],
                [],
                s=12 + 230 * fraction,
                color="grey",
                edgecolors="black",
                linewidths=0.15,
                label=f"{fraction:.0%}",
            )
        )
    ax.legend(
        handles=size_handles,
        title="Raw detection",
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=False,
        fontsize=8,
    )

    ax.set_xlim(-0.7, len(genes) - 0.3)
    ax.set_ylim(len(clusters) - 0.3, -1.8)
    ax.grid(False)

    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


In [8]:
# ---------------------------------------------------------------------
# Balanced marker ranking and raw support
# ---------------------------------------------------------------------
def balanced_marker_subset(
    candidate: ad.AnnData,
) -> ad.AnnData:
    rng = np.random.default_rng(
        RANDOM_STATE
    )
    labels = (
        candidate.obs[CLUSTER_KEY]
        .astype(str)
        .to_numpy()
    )
    selected = []

    for cluster in natural_cluster_order(
        candidate.obs[CLUSTER_KEY]
    ):
        indices = np.flatnonzero(
            labels == cluster
        )
        if len(indices) > MAX_MARKER_CELLS_PER_CLUSTER:
            indices = rng.choice(
                indices,
                size=MAX_MARKER_CELLS_PER_CLUSTER,
                replace=False,
            )
        selected.append(indices)

    selected = np.sort(
        np.concatenate(selected)
    )
    return candidate[selected].copy()


def rank_resolution0p2_markers(
    candidate: ad.AnnData,
) -> pd.DataFrame:
    marker_data = balanced_marker_subset(
        candidate
    )

    sc.tl.rank_genes_groups(
        marker_data,
        groupby=CLUSTER_KEY,
        method="wilcoxon",
        n_genes=min(
            N_MARKER_GENES,
            marker_data.n_vars,
        ),
        pts=True,
        use_raw=False,
        key_added="rank_genes_res0p2",
    )

    table = sc.get.rank_genes_groups_df(
        marker_data,
        group=None,
        key="rank_genes_res0p2",
    )
    table.insert(
        0,
        "resolution",
        PRIMARY_RESOLUTION,
    )

    del marker_data
    gc.collect()
    return table


def raw_support_for_marker_table(
    candidate: ad.AnnData,
    marker_table: pd.DataFrame,
) -> pd.DataFrame:
    marker_table = marker_table.copy()

    genes = [
        gene
        for gene in pd.unique(
            marker_table["names"].astype(str)
        )
        if gene in candidate.var_names
    ]
    gene_index = {
        gene: index
        for index, gene in enumerate(genes)
    }

    raw = candidate[:, genes].layers[
        "counts"
    ]
    raw = (
        raw.tocsr()
        if sp.issparse(raw)
        else sp.csr_matrix(raw)
    )

    labels = (
        candidate.obs[CLUSTER_KEY]
        .astype(str)
        .to_numpy()
    )
    rows = []

    for cluster in natural_cluster_order(
        candidate.obs[CLUSTER_KEY]
    ):
        mask = labels == cluster
        in_group = raw[mask]
        out_group = raw[~mask]

        detection_in = np.asarray(
            (in_group > 0).mean(axis=0)
        ).reshape(-1)
        detection_out = np.asarray(
            (out_group > 0).mean(axis=0)
        ).reshape(-1)
        mean_in = np.asarray(
            in_group.mean(axis=0)
        ).reshape(-1)
        mean_out = np.asarray(
            out_group.mean(axis=0)
        ).reshape(-1)

        for gene in genes:
            index = gene_index[gene]
            rows.append(
                {
                    "group": cluster,
                    "names": gene,
                    "raw_detection_group": float(
                        detection_in[index]
                    ),
                    "raw_detection_reference": float(
                        detection_out[index]
                    ),
                    "raw_detection_delta": float(
                        detection_in[index]
                        - detection_out[index]
                    ),
                    "raw_mean_group": float(
                        mean_in[index]
                    ),
                    "raw_mean_reference": float(
                        mean_out[index]
                    ),
                }
            )

    support = pd.DataFrame(rows)
    result = marker_table.merge(
        support,
        on=["group", "names"],
        how="left",
    )

    result["raw_supported_marker"] = (
        (result["pvals_adj"] <= MARKER_PADJ_MAX)
        & (
            result["logfoldchanges"]
            >= MARKER_LOGFC_MIN
        )
        & (
            result["raw_detection_group"]
            >= MARKER_RAW_DETECTION_MIN
        )
        & (
            result["raw_detection_delta"]
            >= MARKER_RAW_DETECTION_DELTA_MIN
        )
    )
    return result


def top_supported_markers(
    marker_table: pd.DataFrame,
) -> OrderedDict:
    result = OrderedDict()

    for cluster in natural_cluster_order(
        marker_table["group"]
    ):
        subset = marker_table[
            marker_table["group"].astype(str)
            == cluster
        ].copy()

        supported = subset[
            subset["raw_supported_marker"]
        ]
        if len(supported) < TOP_MARKERS_PER_CLUSTER_FOR_DOTPLOT:
            supported = subset[
                (subset["pvals_adj"] <= 0.05)
                & (
                    subset["logfoldchanges"] > 0
                )
            ]

        genes = []
        for gene in supported["names"].astype(str):
            upper = gene.upper()
            if upper.startswith(
                (
                    "MT-",
                    "MT_",
                    "RPS",
                    "RPL",
                    "HB",
                )
            ):
                continue
            if gene not in genes:
                genes.append(gene)
            if len(genes) >= TOP_MARKERS_PER_CLUSTER_FOR_DOTPLOT:
                break

        result[f"Cluster {cluster}"] = genes

    return result


In [9]:
# ---------------------------------------------------------------------
# Cluster composition, UMAP, spatial, and nested-resolution plots
# ---------------------------------------------------------------------
def cluster_composition_table(
    candidate: ad.AnnData,
    column: str,
) -> pd.DataFrame:
    table = (
        candidate.obs.groupby(
            [CLUSTER_KEY, column],
            observed=True,
        )
        .size()
        .rename("n_cells")
        .reset_index()
    )
    table[CLUSTER_KEY] = table[
        CLUSTER_KEY
    ].astype(str)
    table[column] = table[column].astype(str)
    totals = table.groupby(
        CLUSTER_KEY
    )["n_cells"].transform("sum")
    table["fraction_within_cluster"] = (
        table["n_cells"] / totals
    )
    return table.rename(
        columns={CLUSTER_KEY: "cluster"}
    )


def cluster_summary_table(
    candidate: ad.AnnData,
) -> pd.DataFrame:
    rows = []

    for cluster in natural_cluster_order(
        candidate.obs[CLUSTER_KEY]
    ):
        group = candidate.obs[
            candidate.obs[CLUSTER_KEY].astype(str)
            == cluster
        ]

        row = {
            "cluster": cluster,
            "n_cells": int(len(group)),
            "fraction_of_object": float(
                len(group) / candidate.n_obs
            ),
            "n_samples": int(
                group["sample"].nunique()
            ),
            "n_patients": int(
                group["patient"].nunique()
            ),
        }

        for column in [
            "qc_total_counts",
            "qc_n_genes_by_counts",
            "qc_pct_counts_mt",
            "resolvi_diffusion_proportion",
            "S_score",
            "G2M_score",
        ]:
            if column in group:
                row[f"median_{column}"] = float(
                    pd.to_numeric(
                        group[column],
                        errors="coerce",
                    ).median()
                )

        rows.append(row)

    return pd.DataFrame(rows)


def category_color_map(values) -> dict:
    categories = natural_cluster_order(
        values
    )
    cmap = plt.get_cmap(
        "tab20",
        max(len(categories), 2),
    )
    return {
        category: cmap(index)
        for index, category in enumerate(categories)
    }


def downsample_indices(n_cells: int) -> np.ndarray:
    if n_cells <= PLOT_MAX_CELLS:
        return np.arange(n_cells)
    rng = np.random.default_rng(
        RANDOM_STATE
    )
    return np.sort(
        rng.choice(
            n_cells,
            size=PLOT_MAX_CELLS,
            replace=False,
        )
    )


def save_umap_audit(
    candidate: ad.AnnData,
    cancer_type: str,
    path: Path,
):
    indices = downsample_indices(
        candidate.n_obs
    )
    coordinates = candidate.obsm[
        "X_umap"
    ][indices]

    columns = [
        CLUSTER_KEY,
        "sample",
        "biopsy_stage",
        ANNOTATION_COLUMN,
    ]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(15, 13),
    )

    for ax, column in zip(
        axes.flat,
        columns,
    ):
        values = (
            candidate.obs[column]
            .iloc[indices]
            .astype(str)
        )
        mapping = category_color_map(values)

        for category, color in mapping.items():
            mask = values.to_numpy() == category
            ax.scatter(
                coordinates[mask, 0],
                coordinates[mask, 1],
                s=UMAP_POINT_SIZE,
                linewidths=0,
                alpha=0.72,
                color=color,
                label=category,
                rasterized=True,
            )

        if column == CLUSTER_KEY:
            frame = pd.DataFrame(
                {
                    "x": coordinates[:, 0],
                    "y": coordinates[:, 1],
                    "cluster": values.to_numpy(),
                }
            )
            centroids = (
                frame.groupby(
                    "cluster",
                    observed=True,
                )[["x", "y"]]
                .median()
            )
            for cluster, point in centroids.iterrows():
                ax.text(
                    point["x"],
                    point["y"],
                    cluster,
                    fontsize=10,
                    weight="bold",
                    ha="center",
                    va="center",
                )

        ax.set_title(column)
        ax.axis("off")
        ax.legend(
            fontsize=6,
            frameon=False,
            markerscale=4,
            loc="best",
        )

    fig.suptitle(
        f"{cancer_type}: resolution 0.2 tumor/unresolved review",
        fontsize=16,
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_program_umap_grid(
    candidate: ad.AnnData,
    programs: OrderedDict,
    path: Path,
    title: str,
):
    score_pairs = []
    for program in programs:
        column = (
            "program_score__"
            + program_slug(program)
        )
        if column in candidate.obs:
            score_pairs.append(
                (program, column)
            )

    if not score_pairs:
        return

    indices = downsample_indices(
        candidate.n_obs
    )
    coordinates = candidate.obsm[
        "X_umap"
    ][indices]

    n_columns = 3
    n_rows = math.ceil(
        len(score_pairs) / n_columns
    )

    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(
            6 * n_columns,
            5 * n_rows,
        ),
        squeeze=False,
    )

    for ax, (program, column) in zip(
        axes.flat,
        score_pairs,
    ):
        values = candidate.obs[
            column
        ].iloc[indices].to_numpy(dtype=float)

        lower, upper = np.nanquantile(
            values,
            [0.02, 0.98],
        )
        if not np.isfinite(lower):
            lower = np.nanmin(values)
        if not np.isfinite(upper):
            upper = np.nanmax(values)
        if upper <= lower:
            upper = lower + 1e-6

        scatter = ax.scatter(
            coordinates[:, 0],
            coordinates[:, 1],
            c=values,
            s=UMAP_POINT_SIZE,
            cmap="viridis",
            vmin=lower,
            vmax=upper,
            linewidths=0,
            rasterized=True,
        )
        ax.set_title(program)
        ax.axis("off")
        fig.colorbar(
            scatter,
            ax=ax,
            fraction=0.035,
            pad=0.01,
        )

    for ax in axes.flat[
        len(score_pairs):
    ]:
        ax.axis("off")

    fig.suptitle(title, fontsize=16)
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_spatial_maps(
    candidate: ad.AnnData,
    cancer_type: str,
    program_sets: OrderedDict,
    output_dir: Path,
):
    if "spatial" not in candidate.obsm:
        print("Spatial plots skipped: no obsm['spatial'].")
        return

    differentiation_program = next(
        iter(program_sets),
        None,
    )
    differentiation_column = (
        "program_score__"
        + program_slug(
            differentiation_program
        )
        if differentiation_program
        else None
    )
    invasion_column = (
        "program_score__"
        + program_slug(
            "Pan | EMT / mesenchymal"
        )
    )
    hypoxia_column = (
        "program_score__"
        + program_slug(
            "Pan | Hypoxia"
        )
    )

    cluster_mapping = category_color_map(
        candidate.obs[CLUSTER_KEY]
    )

    for sample in sorted(
        candidate.obs["sample"]
        .astype(str)
        .unique()
    ):
        mask = (
            candidate.obs["sample"]
            .astype(str)
            .eq(sample)
            .to_numpy()
        )
        positions = np.flatnonzero(mask)
        coordinates = np.asarray(
            candidate.obsm["spatial"][
                positions
            ],
            dtype=float,
        )

        fig, axes = plt.subplots(
            2,
            2,
            figsize=(13, 12),
        )

        clusters = (
            candidate.obs.iloc[
                positions
            ][CLUSTER_KEY]
            .astype(str)
        )
        colors = [
            cluster_mapping[value]
            for value in clusters
        ]
        axes[0, 0].scatter(
            coordinates[:, 0],
            coordinates[:, 1],
            c=colors,
            s=SPATIAL_POINT_SIZE,
            linewidths=0,
            rasterized=True,
        )
        axes[0, 0].set_title(
            "Resolution 0.2 cluster"
        )

        continuous_panels = [
            (
                axes[0, 1],
                differentiation_column,
                differentiation_program,
            ),
            (
                axes[1, 0],
                invasion_column,
                "Pan | EMT / mesenchymal",
            ),
            (
                axes[1, 1],
                hypoxia_column,
                "Pan | Hypoxia",
            ),
        ]

        for ax, column, title in continuous_panels:
            if column not in candidate.obs:
                ax.axis("off")
                continue
            values = candidate.obs.iloc[
                positions
            ][column].to_numpy(dtype=float)
            scatter = ax.scatter(
                coordinates[:, 0],
                coordinates[:, 1],
                c=values,
                s=SPATIAL_SCORE_POINT_SIZE,
                cmap="viridis",
                linewidths=0,
                rasterized=True,
            )
            ax.set_title(title)
            fig.colorbar(
                scatter,
                ax=ax,
                fraction=0.035,
                pad=0.01,
            )

        for ax in axes.flat:
            ax.set_aspect(
                "equal",
                adjustable="box",
            )
            ax.invert_yaxis()
            ax.axis("off")

        fig.suptitle(
            f"{sample}: {cancer_type} resolution 0.2"
        )
        fig.tight_layout()
        fig.savefig(
            output_dir
            / f"{sample}_resolution0p2_spatial_biology.png",
            dpi=PLOT_DPI,
            bbox_inches="tight",
        )
        plt.close(fig)


def nested_resolution_tables_and_plot(
    candidate: ad.AnnData,
    paths: dict[str, Path],
):
    if COMPARISON_CLUSTER_KEY not in candidate.obs:
        return

    counts = pd.crosstab(
        candidate.obs[CLUSTER_KEY].astype(str),
        candidate.obs[
            COMPARISON_CLUSTER_KEY
        ].astype(str),
    )
    row_fraction = counts.div(
        counts.sum(axis=1),
        axis=0,
    )

    counts.to_csv(
        paths["res02_vs_res06_counts"]
    )
    row_fraction.to_csv(
        paths["res02_vs_res06_fraction"]
    )

    fig, ax = plt.subplots(
        figsize=(
            max(7, 0.6 * counts.shape[1] + 2),
            max(4, 0.65 * counts.shape[0] + 2),
        )
    )
    image = ax.imshow(
        row_fraction.to_numpy(),
        aspect="auto",
        cmap="Blues",
        vmin=0,
        vmax=1,
    )
    ax.set_xticks(
        range(counts.shape[1]),
        labels=counts.columns,
        rotation=90,
    )
    ax.set_yticks(
        range(counts.shape[0]),
        labels=counts.index,
    )
    ax.set_xlabel(
        "Resolution 0.6 cluster"
    )
    ax.set_ylabel(
        "Resolution 0.2 cluster"
    )
    ax.set_title(
        "How each resolution 0.2 cluster splits at resolution 0.6"
    )
    fig.colorbar(
        image,
        ax=ax,
        label="Fraction within resolution 0.2 cluster",
    )
    fig.tight_layout()
    fig.savefig(
        paths["nested_resolution_heatmap"],
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


In [10]:
# ---------------------------------------------------------------------
# Program heatmap and annotation workbench
# ---------------------------------------------------------------------
def save_program_heatmap(
    score_table: pd.DataFrame,
    path: Path,
    title: str,
):
    matrix = score_table.pivot(
        index="cluster",
        columns="program",
        values="mean_program_score",
    )
    matrix = matrix.reindex(
        natural_cluster_order(
            pd.Series(matrix.index)
        )
    )

    fig, ax = plt.subplots(
        figsize=(
            max(11, 0.45 * matrix.shape[1] + 4),
            max(4.5, 0.6 * matrix.shape[0] + 2),
        )
    )
    image = ax.imshow(
        matrix.to_numpy(),
        aspect="auto",
        cmap="RdBu_r",
        vmin=-2,
        vmax=2,
    )
    ax.set_xticks(
        range(matrix.shape[1]),
        labels=matrix.columns,
        rotation=90,
        fontsize=8,
    )
    ax.set_yticks(
        range(matrix.shape[0]),
        labels=matrix.index,
    )
    ax.set_xlabel("")
    ax.set_ylabel("Leiden cluster (resolution 0.2)")
    ax.set_title(title)
    fig.colorbar(
        image,
        ax=ax,
        label="Mean per-cell program score",
    )
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def build_annotation_workbench(
    candidate: ad.AnnData,
    cluster_summary: pd.DataFrame,
    marker_table: pd.DataFrame,
    score_table: pd.DataFrame,
    sample_composition: pd.DataFrame,
    stage_composition: pd.DataFrame,
    source_composition: pd.DataFrame,
) -> pd.DataFrame:
    workbench = cluster_summary.copy()

    top_markers = (
        marker_table[
            marker_table[
                "raw_supported_marker"
            ]
        ]
        .sort_values(
            [
                "group",
                "scores",
            ],
            ascending=[True, False],
        )
        .groupby(
            "group",
            observed=True,
        )["names"]
        .apply(
            lambda values: ";".join(
                pd.unique(
                    values.astype(str)
                )[:15]
            )
        )
        .rename("top_raw_supported_markers")
        .reset_index()
        .rename(columns={"group": "cluster"})
    )
    workbench = workbench.merge(
        top_markers,
        on="cluster",
        how="left",
    )

    program_wide = score_table.pivot(
        index="cluster",
        columns="program",
        values="mean_program_score",
    )
    program_wide.columns = [
        "mean_program__"
        + program_slug(column)
        for column in program_wide.columns
    ]
    workbench = workbench.merge(
        program_wide.reset_index(),
        on="cluster",
        how="left",
    )

    def dominant_category(
        composition: pd.DataFrame,
        category_column: str,
        prefix: str,
    ):
        ordered = composition.sort_values(
            [
                "cluster",
                "fraction_within_cluster",
            ],
            ascending=[True, False],
        )
        top = ordered.groupby(
            "cluster",
            observed=True,
        ).head(1)
        top = top[
            [
                "cluster",
                category_column,
                "fraction_within_cluster",
            ]
        ].rename(
            columns={
                category_column: (
                    f"top_{prefix}"
                ),
                "fraction_within_cluster": (
                    f"top_{prefix}_fraction"
                ),
            }
        )
        return top

    workbench = workbench.merge(
        dominant_category(
            sample_composition,
            "sample",
            "sample",
        ),
        on="cluster",
        how="left",
    )
    workbench = workbench.merge(
        dominant_category(
            stage_composition,
            "biopsy_stage",
            "stage",
        ),
        on="cluster",
        how="left",
    )
    workbench = workbench.merge(
        dominant_category(
            source_composition,
            ANNOTATION_COLUMN,
            "source_label",
        ),
        on="cluster",
        how="left",
    )

    return workbench


In [11]:
# ---------------------------------------------------------------------
# Process one cancer type
# ---------------------------------------------------------------------
def process_cancer_type(
    cancer_type: str,
) -> dict:
    started = time.time()
    paths = output_paths(cancer_type)

    if (
        paths["success"].exists()
        and paths["annotation_workbench"].exists()
        and not OVERWRITE
    ):
        print("Reusing completed deep review:", cancer_type)
        return json.loads(
            paths["success"].read_text()
        )

    paths["failure"].unlink(
        missing_ok=True
    )

    print("\n" + "=" * 90)
    print("Deep review:", cancer_type)
    print("=" * 90)

    candidate = load_cancer_object(
        cancer_type
    )

    programs, availability = (
        audit_program_availability(
            candidate,
            cancer_type,
        )
    )
    availability.to_csv(
        paths["program_availability"],
        index=False,
    )

    print(
        f"{cancer_type}: scoring "
        f"{len(programs)} programs."
    )
    score_columns = score_programs(
        candidate,
        programs,
    )
    score_table, raw_program_table = (
        program_cluster_tables(
            candidate,
            programs,
        )
    )
    score_table.to_csv(
        paths["program_cluster_scores"],
        index=False,
    )
    raw_program_table.to_csv(
        paths["program_raw_detection"],
        index=False,
    )

    marker_table = (
        rank_resolution0p2_markers(
            candidate
        )
    )
    marker_table = raw_support_for_marker_table(
        candidate,
        marker_table,
    )
    marker_table.to_csv(
        paths["marker_table"],
        index=False,
    )
    marker_table[
        marker_table["raw_supported_marker"]
    ].to_csv(
        paths["marker_table_supported"],
        index=False,
    )

    top_marker_groups = (
        top_supported_markers(
            marker_table
        )
    )

    # Composition tables.
    cluster_summary = cluster_summary_table(
        candidate
    )
    sample_composition = cluster_composition_table(
        candidate,
        "sample",
    )
    patient_composition = cluster_composition_table(
        candidate,
        "patient",
    )
    stage_composition = cluster_composition_table(
        candidate,
        "biopsy_stage",
    )
    source_composition = cluster_composition_table(
        candidate,
        ANNOTATION_COLUMN,
    )

    cluster_summary.to_csv(
        paths["cluster_summary"],
        index=False,
    )
    sample_composition.to_csv(
        paths["sample_composition"],
        index=False,
    )
    patient_composition.to_csv(
        paths["patient_composition"],
        index=False,
    )
    stage_composition.to_csv(
        paths["stage_composition"],
        index=False,
    )
    source_composition.to_csv(
        paths["source_label_composition"],
        index=False,
    )

    workbench = build_annotation_workbench(
        candidate,
        cluster_summary,
        marker_table,
        score_table,
        sample_composition,
        stage_composition,
        source_composition,
    )
    workbench.to_csv(
        paths["annotation_workbench"],
        index=False,
    )
    display(workbench)

    # UMAP and spatial plots.
    save_umap_audit(
        candidate,
        cancer_type,
        paths["umap_audit"],
    )

    lineage_programs = OrderedDict(
        (
            program,
            programs[program],
        )
        for program in CANCER_SPECIFIC_PROGRAMS[
            cancer_type
        ]
        if program in programs
    )
    stress_programs = OrderedDict(
        (
            program,
            programs[program],
        )
        for program in PAN_CANCER_PROGRAMS
        if program in programs
    )

    save_program_umap_grid(
        candidate,
        lineage_programs,
        paths["umap_programs_lineage"],
        title=(
            f"{cancer_type}: lineage/differentiation program scores"
        ),
    )
    save_program_umap_grid(
        candidate,
        stress_programs,
        paths["umap_programs_stress"],
        title=(
            f"{cancer_type}: invasion and stress program scores"
        ),
    )

    save_spatial_maps(
        candidate,
        cancer_type,
        lineage_programs,
        paths["spatial"],
    )

    # Hybrid dotplots.
    lineage_dot_table = hybrid_dotplot_table(
        candidate,
        lineage_programs,
    )
    save_hybrid_dotplot(
        lineage_dot_table,
        lineage_programs,
        paths["biology_dotplot_lineage"],
        title=(
            f"{cancer_type}: resolution 0.2 lineage/differentiation "
            "(size = raw detection, color = corrected mean)"
        ),
    )

    stress_dot_table = hybrid_dotplot_table(
        candidate,
        stress_programs,
    )
    save_hybrid_dotplot(
        stress_dot_table,
        stress_programs,
        paths["biology_dotplot_stress"],
        title=(
            f"{cancer_type}: resolution 0.2 invasion/stress "
            "(size = raw detection, color = corrected mean)"
        ),
    )

    top_marker_dot_table = hybrid_dotplot_table(
        candidate,
        top_marker_groups,
    )
    top_marker_dot_table.to_csv(
        paths["top_marker_dotplot_table"],
        index=False,
    )
    save_hybrid_dotplot(
        top_marker_dot_table,
        top_marker_groups,
        paths["top_marker_dotplot"],
        title=(
            f"{cancer_type}: resolution 0.2 top cluster markers "
            "(size = raw detection, color = corrected mean)"
        ),
    )

    save_program_heatmap(
        score_table,
        paths["program_heatmap"],
        title=(
            f"{cancer_type}: mean program score by resolution 0.2 cluster"
        ),
    )

    nested_resolution_tables_and_plot(
        candidate,
        paths,
    )

    if WRITE_CELL_SCORE_PARQUET:
        base_columns = [
            column
            for column in [
                "source_cell_id",
                "sample",
                "patient",
                "biopsy_stage",
                ANNOTATION_COLUMN,
                CLUSTER_KEY,
                COMPARISON_CLUSTER_KEY,
                "qc_total_counts",
                "qc_n_genes_by_counts",
                "qc_pct_counts_mt",
                "resolvi_diffusion_proportion",
                "S_score",
                "G2M_score",
                "phase",
            ]
            if column in candidate.obs
        ]
        score_frame = candidate.obs[
            [
                *base_columns,
                *score_columns,
            ]
        ].copy()
        score_frame.index.name = "cell_id"
        score_frame.to_parquet(
            paths["program_cell_scores"]
        )

    if WRITE_ENRICHED_H5AD:
        candidate.uns[
            "resolution0p2_deep_review"
        ] = {
            "pipeline_version": PIPELINE_VERSION,
            "primary_resolution": PRIMARY_RESOLUTION,
            "cluster_key": CLUSTER_KEY,
            "comparison_resolution": COMPARISON_RESOLUTION,
            "comparison_cluster_key": COMPARISON_CLUSTER_KEY,
            "programs_scored": list(programs),
            "program_score_columns": score_columns,
            "histologic_grade_inferred": False,
        }
        candidate.write_h5ad(
            paths["enriched_h5ad"],
            compression=H5AD_COMPRESSION,
        )

    summary = {
        "completed": True,
        "pipeline_version": PIPELINE_VERSION,
        "cancer_type": cancer_type,
        "n_cells": int(candidate.n_obs),
        "n_genes": int(candidate.n_vars),
        "cluster_key": CLUSTER_KEY,
        "n_clusters": int(
            candidate.obs[CLUSTER_KEY].nunique()
        ),
        "n_programs_scored": int(
            len(programs)
        ),
        "n_raw_supported_markers": int(
            marker_table[
                "raw_supported_marker"
            ].sum()
        ),
        "annotation_workbench": str(
            paths["annotation_workbench"]
        ),
        "program_availability": str(
            paths["program_availability"]
        ),
        "runtime_minutes": float(
            (time.time() - started) / 60
        ),
    }

    atomic_write_json(
        summary,
        paths["summary"],
    )
    atomic_write_json(
        summary,
        paths["success"],
    )

    del candidate
    gc.collect()

    return summary


In [12]:
# ---------------------------------------------------------------------
# Run selected cancer types
# ---------------------------------------------------------------------
RESULTS = {}
FAILURES = {}

for cancer_type in CANCER_TYPES_TO_RUN:
    try:
        RESULTS[cancer_type] = (
            process_cancer_type(
                cancer_type
            )
        )
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        FAILURES[cancer_type] = error

        paths = output_paths(
            cancer_type
        )
        atomic_write_json(
            {
                "completed": False,
                "pipeline_version": PIPELINE_VERSION,
                "cancer_type": cancer_type,
                "error": error,
                "traceback": traceback.format_exc(),
            },
            paths["failure"],
        )

        traceback.print_exc(limit=30)

        if not CONTINUE_ON_ERROR:
            raise
    finally:
        plt.close("all")
        gc.collect()

summary = pd.DataFrame(
    RESULTS.values()
)
summary.to_csv(
    OUTPUT_ROOT
    / "all_cancer_types_resolution0p2_deep_review_summary.csv",
    index=False,
)
atomic_write_json(
    FAILURES,
    OUTPUT_ROOT
    / "all_cancer_types_resolution0p2_deep_review_failures.json",
)

print("Completed:", sorted(RESULTS))
print("Failures:", json.dumps(FAILURES, indent=2))

if FAILURES:
    raise RuntimeError(
        "At least one cancer-type deep review failed. "
        "Completed outputs remain reusable."
    )



Deep review: melanoma
Loading Zarr: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2/melanoma/melanoma_tumor_unresolved_multires_leiden.zarr
AnnData object with n_obs × n_vars = 521397 × 14887
    obs: 'source_cell_id', 'sample', 'patient', 'cancer_type', 'biopsy_stage', 'prelim_cell_type_primary_tumor_expanded', 'prelim_T_confidence_tier', 'prelim_T_confidence_rank', 'prelim_Treg_confidence_tier', 'prelim_Treg_confidence_rank', 'prelim_T_primary', 'prelim_CD4_T', 'prelim_CD8_T', 'prelim_Treg_supported', 'prelim_Treg_high_confidence', 'rescue_T_mixing_status', 'rescue_T_percentile_minus_max_nonT', 'tumor_epithelial_confidence_tier', 'mt_high', 'low_genes', 'qc_pass', 'S_score', 'G2M_score', 'phase', 'cell_cycle_scored', 'resolvi_diffusion_proportion', 'qc_total_counts', 'qc_n_genes_by_counts', 'qc_pct_counts_mt', 'qc_pct_counts_ribo', 'qc_pct_counts_hb', 'source_cancer_type_b

,cluster,n_cells,fraction_of_object,n_samples,n_patients,median_qc_total_counts,median_qc_n_genes_by_counts,median_qc_pct_counts_mt,median_resolvi_diffusion_proportion,median_S_score,...,mean_program__pan_oxidative_nrf2_associated,mean_program__pan_proliferation,mean_program__pan_upr_er_stress,mean_program__pan_p53_dna_damage,top_sample,top_sample_fraction,top_stage,top_stage_fraction,top_source_label,top_source_label_fraction
0,0,6460,0.012390,4,2,29.0,28.0,9.523809,0.011000,-0.017201,...,0.277681,-0.213609,0.070340,0.373262,C2D15_30_16,0.676935,C2D15,0.678019,Other/unresolved,0.902632
1,1,14556,0.027917,6,3,204.0,168.0,2.272727,0.011018,-0.009761,...,-0.145174,-0.653852,-0.119762,-0.195506,Screen_18_23,0.796991,Screen,0.798708,Other/unresolved,0.622355
2,2,41924,0.080407,6,3,94.0,75.0,2.479339,0.011000,-0.019026,...,0.227459,-0.170345,0.471303,0.299387,C2D15_30_16,0.597033,C2D15,0.603998,Other/unresolved,0.999022
3,3,17468,0.033502,6,3,156.0,133.0,10.975610,0.011000,-0.014012,...,0.244456,0.903742,-0.015866,-0.035204,C2D15_30_16,0.602301,C2D15,0.777880,Other/unresolved,0.916762
4,4,45756,0.087757,6,3,140.0,123.0,3.030303,0.011000,-0.020089,...,-0.415195,-0.383219,-0.452196,0.090523,Screen_16_22,0.509988,Screen,0.523254,Other/unresolved,0.989291
5,5,7745,0.014854,3,2,264.0,214.0,1.347709,0.011000,-0.025973,...,-0.164543,-0.229253,-0.823948,0.811315,Screen_16_22,0.673338,Screen,0.673596,Other/unresolved,0.998192
6,6,35020,0.067166,6,3,102.0,92.0,11.000000,0.011000,0.000000,...,0.424867,2.291314,0.047548,0.034487,C2D15_30_16,0.894032,C2D15,0.895774,Other/unresolved,0.977299
7,7,29496,0.056571,6,3,107.0,96.0,9.728678,0.011000,-0.025511,...,0.333303,0.259880,0.446513,0.358805,C2D15_30_16,0.726370,C2D15,0.726539,Other/unresolved,0.890019
8,8,8272,0.015865,6,3,454.5,391.0,4.878049,0.011000,-0.033164,...,-0.364133,-0.379876,-0.596702,0.080437,C2D15_16_22,0.571809,C2D15,0.574347,Tumor/epithelial,0.690401
9,9,3497,0.006707,4,2,464.0,361.0,1.075269,0.011000,-0.013700,...,-0.111777,-0.843173,-0.036366,0.188809,Screen_18_23,0.855876,Screen,0.856162,Other/unresolved,0.873606



Deep review: NSCLC
Loading Zarr: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2/nsclc/nsclc_tumor_unresolved_multires_leiden.zarr
AnnData object with n_obs × n_vars = 140501 × 14887
    obs: 'source_cell_id', 'sample', 'patient', 'cancer_type', 'biopsy_stage', 'prelim_cell_type_primary_tumor_expanded', 'prelim_T_confidence_tier', 'prelim_T_confidence_rank', 'prelim_Treg_confidence_tier', 'prelim_Treg_confidence_rank', 'prelim_T_primary', 'prelim_CD4_T', 'prelim_CD8_T', 'prelim_Treg_supported', 'prelim_Treg_high_confidence', 'rescue_T_mixing_status', 'rescue_T_percentile_minus_max_nonT', 'tumor_epithelial_confidence_tier', 'mt_high', 'low_genes', 'qc_pass', 'S_score', 'G2M_score', 'phase', 'cell_cycle_scored', 'resolvi_diffusion_proportion', 'qc_total_counts', 'qc_n_genes_by_counts', 'qc_pct_counts_mt', 'qc_pct_counts_ribo', 'qc_pct_counts_hb', 'source_cancer_type_before_sel

,cluster,n_cells,fraction_of_object,n_samples,n_patients,median_qc_total_counts,median_qc_n_genes_by_counts,median_qc_pct_counts_mt,median_resolvi_diffusion_proportion,median_S_score,...,mean_program__pan_oxidative_nrf2_associated,mean_program__pan_proliferation,mean_program__pan_upr_er_stress,mean_program__pan_p53_dna_damage,top_sample,top_sample_fraction,top_stage,top_stage_fraction,top_source_label,top_source_label_fraction
0,0,13141,0.093530,4,2,1981.0,1402.0,6.216313,0.114167,0.140382,...,0.318189,1.913689,-0.023670,-0.231527,C2D15_17_26,0.751084,C2D15,0.751160,Other/unresolved,0.872993
1,1,1096,0.007801,3,2,333.5,250.0,2.633230,0.132987,-0.039350,...,-0.991638,-0.581078,-0.317524,0.728857,C2D15_39_21,0.995438,C2D15,0.998175,Tumor/epithelial,0.971715
2,2,57750,0.411029,4,2,1264.0,959.0,5.212433,0.011097,-0.033412,...,0.140489,-0.268901,0.308231,-0.179817,C2D15_17_26,0.655758,C2D15,0.655948,Other/unresolved,0.688831
3,3,12340,0.087829,4,2,284.0,228.0,5.588289,0.012289,-0.019460,...,-0.197265,-0.313188,-0.386133,-0.078103,C2D15_17_26,0.647083,C2D15,0.663047,Other/unresolved,0.991086
4,4,18537,0.131935,4,2,343.0,230.0,2.586207,0.083575,-0.021708,...,-0.349325,-0.390811,-0.131257,0.168003,C2D15_17_26,0.486756,C2D15,0.657550,Other/unresolved,0.993365
5,5,14591,0.103850,4,2,299.0,231.0,2.473498,0.135211,-0.042932,...,-0.084915,-0.160641,-0.217464,0.616270,Screen_39_21,0.748544,Screen,0.749572,Other/unresolved,0.863820
6,6,1262,0.008982,4,2,766.0,565.0,8.227633,0.127735,-0.016499,...,0.072200,-0.027271,-0.073738,-0.403289,C2D15_17_26,0.749604,C2D15,0.751189,Other/unresolved,0.990491
7,7,3929,0.027964,2,1,331.0,293.0,5.022537,0.220390,0.177238,...,-0.352242,2.354747,-0.152418,1.150001,Screen_39_21,0.999745,Screen,0.999745,Other/unresolved,0.625859
8,8,8850,0.062989,4,2,1494.0,1025.5,8.881950,0.014389,-0.018387,...,0.183372,-0.128610,-0.022190,-0.511818,C2D15_17_26,0.654350,C2D15,0.656384,Other/unresolved,0.966780
9,9,9005,0.064092,4,2,534.0,432.0,3.010753,0.025438,-0.037329,...,-0.154342,-0.401133,-0.653227,0.220395,C2D15_17_26,0.533259,C2D15,0.557246,Other/unresolved,0.993115



Deep review: colon_cancer
Loading Zarr: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/16_tumor_unresolved_leiden_multiresolution_selectionfix_v2/colon_cancer/colon_cancer_tumor_unresolved_multires_leiden.zarr
AnnData object with n_obs × n_vars = 26630 × 14887
    obs: 'source_cell_id', 'sample', 'patient', 'cancer_type', 'biopsy_stage', 'prelim_cell_type_primary_tumor_expanded', 'prelim_T_confidence_tier', 'prelim_T_confidence_rank', 'prelim_Treg_confidence_tier', 'prelim_Treg_confidence_rank', 'prelim_T_primary', 'prelim_CD4_T', 'prelim_CD8_T', 'prelim_Treg_supported', 'prelim_Treg_high_confidence', 'rescue_T_mixing_status', 'rescue_T_percentile_minus_max_nonT', 'tumor_epithelial_confidence_tier', 'mt_high', 'low_genes', 'qc_pass', 'S_score', 'G2M_score', 'phase', 'cell_cycle_scored', 'resolvi_diffusion_proportion', 'qc_total_counts', 'qc_n_genes_by_counts', 'qc_pct_counts_mt', 'qc_pct_counts_ribo', 'qc_pct_counts_hb', 'source_ca

,cluster,n_cells,fraction_of_object,n_samples,n_patients,median_qc_total_counts,median_qc_n_genes_by_counts,median_qc_pct_counts_mt,median_resolvi_diffusion_proportion,median_S_score,...,mean_program__pan_oxidative_nrf2_associated,mean_program__pan_proliferation,mean_program__pan_upr_er_stress,mean_program__pan_p53_dna_damage,top_sample,top_sample_fraction,top_stage,top_stage_fraction,top_source_label,top_source_label_fraction
0,0,2167,0.081374,2,1,467.0,390.0,8.459596,0.011001,-0.014705,...,-0.426558,0.432586,-0.539146,-0.232176,Screen_23_25,0.998154,Screen,0.998154,Other/unresolved,0.881864
1,1,4450,0.167105,2,1,95.5,84.0,7.936508,0.011079,-0.011526,...,0.053474,0.340105,-0.173939,0.053797,C2D15_23_25,0.965843,C2D15,0.965843,Other/unresolved,0.789213
2,2,3129,0.117499,2,1,301.0,265.0,2.205882,0.011001,-0.031003,...,0.605550,-0.614859,1.018152,0.088901,Screen_23_25,0.999680,Screen,0.999680,Other/unresolved,0.673058
3,3,2010,0.075479,2,1,106.0,98.5,1.749126,0.011002,-0.016520,...,0.335739,-0.526308,0.467490,0.099936,Screen_23_25,0.997015,Screen,0.997015,Other/unresolved,0.997015
4,4,632,0.023733,2,1,114.0,103.0,2.020202,0.011641,-0.015743,...,0.467562,-0.801261,0.188682,0.036409,C2D15_23_25,0.669304,C2D15,0.669304,Other/unresolved,0.677215
5,5,2006,0.075329,2,1,310.0,273.5,3.634421,0.011001,-0.028499,...,0.386867,-0.365408,0.518493,-0.082801,Screen_23_25,0.987537,Screen,0.987537,Other/unresolved,0.622134
6,6,6193,0.232557,2,1,691.0,560.0,7.320644,0.011001,-0.013742,...,0.015617,0.791014,-0.173376,-0.183542,Screen_23_25,0.999839,Screen,0.999839,Other/unresolved,0.674148
7,7,2600,0.097634,2,1,49.0,43.0,1.538462,0.011002,0.000000,...,-0.678864,-0.675474,-0.365991,0.342841,C2D15_23_25,0.976923,C2D15,0.976923,Other/unresolved,0.978077
8,8,3383,0.127037,1,1,136.0,118.0,3.724395,0.011037,-0.016162,...,-0.354103,-0.388755,-0.377455,0.073153,Screen_23_25,1.000000,Screen,1.000000,Other/unresolved,0.998226
9,9,60,0.002253,2,1,22.0,21.0,0.000000,0.011000,0.000000,...,-1.476955,-0.975603,-0.670503,-1.254146,Screen_23_25,0.900000,Screen,0.900000,Other/unresolved,1.000000


Completed: ['NSCLC', 'colon_cancer', 'melanoma']
Failures: {}


# How to interpret the main outputs

## 1. Start with the annotation workbench

```text
tables/<cancer>_resolution0p2_cluster_annotation_workbench.csv
```

For every cluster, it combines:

- cluster size;
- sample and patient breadth;
- dominant sample and biopsy stage;
- original `Tumor/epithelial` versus `Other/unresolved` composition;
- median QC and ResolVI-diffusion values when available;
- top raw-supported markers;
- mean exploratory program scores.

## 2. Use the hybrid dotplots

The custom dotplots deliberately separate two kinds of evidence:

```text
dot size  = fraction of cells with observed raw Proseg counts > 0
dot color = mean ResolVI-corrected log-expression, z-scored within each gene
```

This avoids the misleading behavior of using dense corrected expression for
both dot size and color.

## 3. Review resolution 0.2 versus 0.6

```text
figures/<cancer>_resolution0p2_vs_resolution0p6_heatmap.png
```

Rows show resolution 0.2 clusters and columns show the resolution 0.6
subclusters nested within them. This can help identify whether a broad 0.2
state is biologically coherent or merely merges several unrelated 0.6 states.

## 4. Differentiation status

Interpret differentiation as a continuum of lineage-associated programs:

```text
lineage-high / differentiated-like
transitory or lineage-plastic
stem/progenitor-like
dedifferentiated / EMT-associated
```

Do not convert these directly into pathology grades such as “well,”
“moderately,” or “poorly differentiated” without histologic review.

## 5. Competing-lineage audit

The candidate universe intentionally includes `Other/unresolved`. A cluster
with strong immune, fibroblast, or endothelial audit genes may represent a
remaining non-cancer population, segmentation mixing, or a true tumor program
that requires manual review.


# Selected references informing the exploratory programs

- Tirosh I, et al. *Dissecting the multicellular ecosystem of metastatic
  melanoma by single-cell RNA-seq.* Science. 2016;352:189–196.
  doi:10.1126/science.aad0501.
- Tsoi J, et al. *Multi-stage Differentiation Defines Melanoma Subtypes with
  Differential Vulnerability to Drug-Induced Iron-Dependent Oxidative Stress.*
  Cancer Cell. 2018;33:890–904.e5.
  doi:10.1016/j.ccell.2018.03.017.
- Wang Z, et al. *Deciphering cell lineage specification of human lung
  adenocarcinoma with single-cell RNA sequencing.* Nature Communications.
  2021;12:6500. doi:10.1038/s41467-021-26770-2.
- De Zuani M, et al. *Single-cell and spatial transcriptomics analysis of
  non-small cell lung cancer.* Nature Communications. 2024;15:4388.
  doi:10.1038/s41467-024-48700-8.
- Uhlitz F, et al. *A census of cell types and paracrine interactions in
  colorectal cancer.* bioRxiv. 2020. doi:10.1101/2020.01.10.901579.
- Lee S, et al. *Network Inference Analysis Identifies SETDB1 as a Key
  Regulator for Reverting Colorectal Cancer Cells into Differentiated
  Normal-Like Cells.* Molecular Cancer Research. 2020;18:118–129.
  doi:10.1158/1541-7786.MCR-19-0450.

The stress programs are compact exploratory representations of canonical
hypoxia, AP-1/immediate-early, heat-shock, unfolded-protein, oxidative,
DNA-damage, and interferon responses. They are intended for cluster review, not
as clinical biomarkers.
